Célula única da Gold

In [0]:
from pyspark.sql import functions as F

silver = spark.table("workspace.process_sigma.silver_orders")

# Gold OTD: período × região
gold_otd = (
    silver.groupBy("period", F.col("order_region").alias("region"))
    .agg(
        F.count("*").alias("total_deliveries"),
        F.sum(F.when(~F.col("is_late"), 1).otherwise(0)).alias("on_time"),
    )
    .withColumn("otd_pct", F.round(F.col("on_time") / F.col("total_deliveries") * 100, 2))
)

# dpmo_to_sigma e classify_sigma, copiadas do notebook
def sigma_expr(d):
    return (
        F.when(d <= 0, 6.0)
        .when(d >= 999999, 0.5)
        .when(d >= 500000, 1.5)
        .when(d >= 308538, 2.0)
        .when(d >= 66807, 3.0)
        .when(d >= 6210, 4.0)
        .when(d >= 233, 5.0)
        .otherwise(F.round(0.8406 + F.sqrt(F.greatest(29.37 - 2.221 * F.log(d), F.lit(0.0))), 2))
    )

def classify_expr(s):
    return (
        F.when(s >= 6, "World Class")
        .when(s >= 5, "Excellent")
        .when(s >= 4, "Good")
        .when(s >= 3, "Attention")
        .otherwise("Critical")
    )

# Gold Sigma: período
gold_sigma = (
    silver.groupBy("period")
    .agg(
        F.count("*").alias("total_opportunities"),
        F.sum(
            F.col("is_late").cast("int")
            + F.col("damage_flag").cast("int")
            + F.col("return_flag").cast("int")
        ).alias("total_defects"),
    )
    .withColumn("dpmo", F.round(F.col("total_defects") / F.col("total_opportunities") * 1_000_000, 2))
    .withColumn("sigma_level", sigma_expr(F.col("dpmo")))
    .withColumn("classification", classify_expr(F.col("sigma_level")))
)

gold_otd.write.mode("overwrite").saveAsTable("workspace.process_sigma.gold_otd")
gold_sigma.write.mode("overwrite").saveAsTable("workspace.process_sigma.gold_sigma")

Conferir e ver o resumo global

In [0]:
g_otd = spark.table("workspace.process_sigma.gold_otd")
g_sig = spark.table("workspace.process_sigma.gold_sigma")

print(g_otd.count(), g_sig.count())
g_otd.orderBy("period", "region").show(6, truncate=False)
g_sig.orderBy("period").show(truncate=False)

tot = g_otd.agg(F.sum("on_time").alias("ot"), F.sum("total_deliveries").alias("n")).first()
print("OTD global:", round(tot["ot"] / tot["n"] * 100, 2))

45 9
+-------+-------------+----------------+-------+-------+
|period |region       |total_deliveries|on_time|otd_pct|
+-------+-------------+----------------+-------+-------+
|2022-01|ASIA         |16              |10     |62.5   |
|2022-01|EUROPE       |9               |3      |33.33  |
|2022-01|NORTH AMERICA|12              |4      |33.33  |
|2022-01|OCEANIA      |12              |6      |50.0   |
|2022-01|SOUTH AMERICA|13              |8      |61.54  |
|2022-02|ASIA         |12              |8      |66.67  |
+-------+-------------+----------------+-------+-------+
only showing top 6 rows
+-------+-------------------+-------------+---------+-----------+--------------+
|period |total_opportunities|total_defects|dpmo     |sigma_level|classification|
+-------+-------------------+-------------+---------+-----------+--------------+
|2022-01|62                 |31           |500000.0 |1.5        |Critical      |
|2022-02|56                 |27           |482142.86|2.0        |Critical    